# Split the layers, then keep them busy

The transformer has more layers than one device can hold. In the previous notebook,
TP split computations inside a layer; FSDP gathered parameters before using them.
There is another boundary already present in the model: one layer follows another.

Put successive layers on different devices. Each **pipeline stage** owns its parameters
and receives activations from the preceding stage. Weights stay in place. This can be
useful across groups with expensive communication, while TP operates within a group.
It does not guarantee a speedup: a stage cannot compute an activation it has not received.

We will first build a forward pipeline, then train through it. You need the meaning of
forward, backward, and activation gradients; you do not need a distributed autograd
framework. Flock runs the workers on one CPU process. The schedules and byte counts
below are models of dependencies and resource use, not timing measurements.

In [1]:
import flock
import flock.torch as ft
import torch

from dist_puzzles import check
from dist_puzzles.lessons.pipeline import check_pipeline_1f1b, check_pipeline_forward

## 1. Layers stay with their owners

For two stages, $h=f_0(X)$ lives first on rank 0, then travels to rank 1 so it can
compute $Y=f_1(h)$. For more stages, repeat the same dependency along adjacent ranks.
Only the last rank returns the full output.

If a boundary activation has shape $[B,S,D]$ in BF16, it carries $2BSD$ bytes. During
training its gradient also travels in the opposite direction (assuming the same dtype).
The count is per boundary, not multiplied by every layer inside a stage.

In [2]:
batch, sequence, width, stages = 1, 8192, 4096, 4
boundary_bytes = batch * sequence * width * 2
print(f"One boundary activation: {boundary_bytes / 2**20:.0f} MiB")
print(f"Forward across {stages - 1} boundaries: {(stages - 1) * boundary_bytes / 2**20:.0f} MiB")
print(f"Forward + backward boundary payloads: {2 * (stages - 1) * boundary_bytes / 2**20:.0f} MiB")

One boundary activation: 64 MiB
Forward across 3 boundaries: 192 MiB
Forward + backward boundary payloads: 384 MiB


Splitting a fixed batch into microbatches leaves these ideal payload bytes unchanged,
but increases message count. Smaller transfers are not automatically faster: startup
latency and less efficient small matrix multiplications can matter.

### Puzzle 1 — forward through adjacent stages

Each rank receives one local `stage`. Only rank 0 receives `X:[B,D]`; other ranks
receive `None`. Split the batch into `microbatch_count` equal nonempty pieces and
forward each piece through the stages in original batch order. Use adjacent links
only; the last rank returns concatenated outputs, and other ranks return `None`.

Every send must finish. The tiny stages below use Linear and GELU; they operate on
examples independently. This equivalence would need more care for batch-dependent
operations. No parameters are sent.

In [3]:
async def pipeline_forward(stage, X, microbatch_count):
    rank, world = flock.get_rank(), flock.get_world_size()
    inputs = X.chunk(microbatch_count) if rank == 0 else None
    results = []
    for microbatch in range(microbatch_count):
        activation = inputs[microbatch] if rank == 0 else await ft.recv(rank - 1, tag=microbatch)
        output = stage(activation)
        if rank + 1 < world:
            await ft.send(rank + 1, output, tag=microbatch)
        else:
            results.append(output)
    return torch.cat(results) if rank == world - 1 else None

In [4]:
torch.manual_seed(0)
stages = [torch.nn.Linear(2, 2), torch.nn.Linear(2, 1)]
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])

run = flock.distribute(workers=2, seed=0)(pipeline_forward)
results = run(flock.per_rank(stages), flock.per_rank([x, None]), microbatch_count=2)
check(results, [None, stages[1](stages[0](x))], rtol=1e-5, atol=1e-5)
results[-1]

tensor([[-0.8730],
        [-2.1196]], grad_fn=<CatBackward0>)

In [5]:
check_pipeline_forward(pipeline_forward)

All checks passed.


## 2. Enough memory, but idle stages

With one batch, stage 1 waits while stage 0 computes; stage 0 is idle once it has
passed that batch on. Microbatches let stage 0 work on a later piece while stage 1
works on an earlier piece.

For this diagram only, assume each stage takes one time slot per microbatch and
communication takes no time. A dot is an idle slot. Numbered entries identify
microbatches, not ranks.

In [6]:
depth, microbatches = 4, 4
for rank in range(depth):
    slots = [
        str(t - rank) if rank <= t < rank + microbatches else "." for t in range(microbatches + depth - 1)
    ]
    print(f"stage {rank}: " + "  ".join(slots))
print()
for count in (1, 4, 16):
    utilization = count / (count + depth - 1)
    print(
        f"{count:2} microbatches: {utilization:.1%} ideal forward utilization; "
        f"{1 - utilization:.1%} bubble fraction"
    )

stage 0: 0  1  2  3  .  .  .
stage 1: .  0  1  2  3  .  .
stage 2: .  .  0  1  2  3  .
stage 3: .  .  .  0  1  2  3

 1 microbatches: 25.0% ideal forward utilization; 75.0% bubble fraction
 4 microbatches: 57.1% ideal forward utilization; 42.9% bubble fraction
16 microbatches: 84.2% ideal forward utilization; 15.8% bubble fraction


There are $MP$ useful stage-slots out of $P(M+P-1)$ available slots, so ideal forward
utilization is $M/(M+P-1)$. The fill and drain gaps form the **pipeline bubble**.
This model assumes equal stages and ignores network cost. It does not predict wall-clock
speed when changing microbatch size. A slow stage, uneven layers, or a costly boundary
can leave more devices waiting.

## 3. Backward must cross the boundary too

Sending an activation to a different process does not send its autograd graph. At a
stage boundary, receive a value and make it a new leaf requiring gradients. Later,
receive the output gradient, run local backward, and send the input gradient upstream.

For a last-stage mean loss, each of $M$ equal microbatches contributes `loss / M`.
Accumulate parameter gradients across all microbatches, then apply **one** optimizer
step. Updating a stage early would make different microbatches use different weights.

We could do every forward first and then every backward. That holds many activation
graphs at once. A **one-forward, one-backward (1F1B)** schedule starts backward earlier,
so graphs can be released as their microbatches finish. It does not make bubbles vanish.

### Puzzle 2 — a non-interleaved 1F1B training step

Rank $r$ owns a stage; rank 0 uses the inputs and the last rank uses the targets.
For a convenient harness, `X` and `target` are passed to all ranks, but intermediate
stages must compute only from received activations. This exercise does not measure
input-storage savings.

With $P$ stages and $M$ microbatches, first perform $\min(P-r-1,M)$ warmup forwards.
Then alternate one forward and one backward, and finally drain the remaining backwards.
Use each stage's oldest outstanding microbatch for backward. The returned event list
records local completion order, not a hardware timeline.

Return `(events, parameters, output)`: updated local parameter copies on every rank,
and the concatenated **pre-update** output only on the last rank. Communicate only on
adjacent links. Keep every Work and complete it before returning. Batch size must be
positive and divisible by the microbatch count; the example has no dropout.

<details><summary>Hint: opposite directions can wait on one another</summary>
A stage may need to post a forward send before receiving an older microbatch's backward
message. Immediately waiting on every send can prevent the neighbour from reaching the
matching receive. Keep the pending Work and wait when the schedule permits it.
</details>

In [7]:
async def pipeline_1f1b(
    stage: torch.nn.Module,
    X: torch.Tensor,
    target: torch.Tensor,
    microbatch_count: int,
    learning_rate: float,
) -> tuple[list[tuple[str, int]], tuple[torch.Tensor, ...], torch.Tensor | None]:
    rank = flock.get_rank()
    world_size = flock.get_world_size()
    if microbatch_count < 1:
        raise ValueError(f"microbatch_count must be positive, got {microbatch_count}")
    if target.shape[0] != X.shape[0]:
        raise ValueError(f"input and target batch sizes must match, got {X.shape[0]} and {target.shape[0]}")
    if X.shape[0] == 0 or X.shape[0] % microbatch_count:
        raise ValueError(
            f"batch size {X.shape[0]} must be positive and divisible by microbatch_count={microbatch_count}"
        )

    inputs = X.chunk(microbatch_count)
    targets = target.chunk(microbatch_count)
    optimizer = torch.optim.SGD(stage.parameters(), lr=learning_rate)
    optimizer.zero_grad(set_to_none=True)

    activations: dict[int, torch.Tensor] = {}
    outputs: dict[int, torch.Tensor] = {}
    final_outputs: dict[int, torch.Tensor] = {}
    events: list[tuple[str, int]] = []
    pending_sends: list[flock.Work[None]] = []

    async def forward(microbatch: int) -> None:
        if rank == 0:
            activation = inputs[microbatch]
        else:
            received = await ft.recv(rank - 1, tag=microbatch)
            activation = received.detach().requires_grad_()

        output = stage(activation)
        activations[microbatch] = activation
        outputs[microbatch] = output

        if rank == world_size - 1:
            final_outputs[microbatch] = output.detach().clone()
        else:
            pending_sends.append(ft.isend(rank + 1, output.detach(), tag=microbatch))

        events.append(("F", microbatch))

    async def backward(microbatch: int) -> None:
        output = outputs.pop(microbatch)
        activation = activations.pop(microbatch)

        if rank == world_size - 1:
            loss = torch.nn.functional.mse_loss(output, targets[microbatch])
            (loss / microbatch_count).backward()
        else:
            output_grad = await ft.recv(rank + 1, tag=microbatch)
            output.backward(output_grad)

        if rank > 0:
            activation_grad = activation.grad
            assert activation_grad is not None
            pending_sends.append(ft.isend(rank - 1, activation_grad, tag=microbatch))

        events.append(("B", microbatch))

    warmup = min(world_size - rank - 1, microbatch_count)
    remaining = microbatch_count - warmup

    for microbatch in range(warmup):
        await forward(microbatch)

    for microbatch in range(remaining):
        await forward(microbatch + warmup)
        await backward(microbatch)

    for microbatch in range(remaining, microbatch_count):
        await backward(microbatch)

    for send in pending_sends:
        await send.wait()

    optimizer.step()
    parameters = tuple(parameter.detach().clone() for parameter in stage.parameters())
    output = (
        torch.cat([final_outputs[microbatch] for microbatch in range(microbatch_count)])
        if rank == world_size - 1
        else None
    )
    return events, parameters, output

### Try one update

Each stage owns its weights; only the boundary activations and gradients are exchanged.
The example keeps the batch tiny enough to inspect. The checker below compares all
updated parameters with a sequential training step, not just the final output.

In [8]:
torch.manual_seed(0)
stages = [torch.nn.Linear(2, 2), torch.nn.Linear(2, 1)]
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
target = torch.tensor([[1.0], [0.0]])

run = flock.distribute(workers=2, seed=0)(pipeline_1f1b)
results = run(flock.per_rank(stages), x, target, microbatch_count=2, learning_rate=0.01)
for rank, (events, _, _) in enumerate(results):
    print(f"stage {rank}: {events}")

stage 0: [('F', 0), ('F', 1), ('B', 0), ('B', 1)]
stage 1: [('F', 0), ('B', 0), ('F', 1), ('B', 1)]


In [9]:
check_pipeline_1f1b(pipeline_1f1b)

All checks passed.


Checks include one stage, fewer microbatches than stages, and several worker schedules.
They compare pre-update outputs, updated parameters, adjacent links, and local 1F1B order.
That exact local order is required here because implementing this schedule is the task;
it would not be appropriate for a puzzle allowing any valid pipeline schedule.

## What the schedule changed

The stage weights stayed in place. The communication carried boundary activations and
their gradients. Microbatches supplied independent pieces of work to fill the pipeline;
1F1B changed how long forward graphs had to stay alive before their backward.

This implementation keeps pending send handles until the end for clarity. A memory-aware
runtime should retire completed transfers promptly. Flock's tracer and tensor copies are
not a measurement of device activation memory. Likewise, posting a nonblocking transfer
exposes an opportunity for overlap; it does not demonstrate actual network/compute overlap.

On real hardware, stage balance, transfer latency, activation size, and kernel efficiency
all affect the result. More stages reduce local layers but can increase idle time. More
microbatches can fill bubbles but introduce more messages and smaller compute kernels.
The [Ultra-Scale Playbook](https://huggingface.co/spaces/nanotron/ultrascale-playbook)
and [How To Scale Your Model](https://jax-ml.github.io/scaling-book/training/) develop
these trade-offs further.

The next step toward a complete framework is composition: each stage may itself use TP,
CP, EP, and sharded state on different groups. Its gradients and update must still match
the same single-device computation.